# Customer Analysis

## Goal

1. How is the customer base structured (customer type, account status, primary freight type, booking type, load type)? (*section 3*)
2. How much does revenue per load vary between customers, is that more than chance would explain, and does `annual_revenue_potential` predict actual revenue? (*4.1, 4.2, 4.3*)
3. Does revenue differ by booking type, load type, customer type or credit terms, also after accounting for distance, and does a customer's primary freight type match the load type? (*section 5*)
4. Does delivery service level (on-time rate, detention) differ by customer, or is the spread consistent with chance? (*6.1, 6.2*)
5. How concentrated is revenue across customers? (*section 7*)

## Data Basis

**Via pipeline** (`clean_customer_analysis.csv`, one row per load):
`loads`, `customers` (joined via `customer_id`: `customer_name`, `customer_type`, `credit_terms_days`, `primary_freight_type`, `account_status`, `contract_start_date`, `annual_revenue_potential`), `routes`, `trips` and the pickup/delivery events (pivoted into `pickup_*` / `delivery_*` columns), plus the delivery metrics.

**Loaded directly in this notebook:**
- `customers`, `loads`: raw tables for the data quality checks
- `delivery_events`: raw events, one row per event, restricted to the loads kept by the pipeline (for the service level analysis in section 6)

**Derived in this notebook:** `total_billed` (revenue + fuel surcharge) and revenue per mile, as in notebook 4.

## Key Findings
- Customers hardly differ in revenue: mean revenue per load varies by only 3 % between customers, and a permutation test shows that this is no more than chance (p = 0.39; per mile p = 0.59)
- `annual_revenue_potential` does not predict actual revenue (Spearman rho = −0.08, p = 0.27); every customer bills about 438,000 $ per year, the median customer only about 16 % of its potential
- Revenue per mile is the same in all segments (booking type, load type, customer type, credit terms; max. difference 0.4 %); the fuel surcharge is rate × typical distance, so its strong correlation with revenue (r = 0.85) is a distance effect
- Service level does not differ between customers beyond chance: on-time rates 50.0–60.7 % (permutation test p = 0.77), detention 85.9–98.8 min (p = 0.58); customer type and account status make no difference
- Revenue is spread almost evenly: the top 10 % of customers generate 11.2 % of the revenue, the largest customer 0.59 %
- 16 % of the loads belong to customers with an inactive account; they are kept because the data does not show when the status changed

## 1. Load Data (via Pipeline)

In [ ]:
import sys

sys.path.append("../src")
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="darkgrid")

from scipy import stats

from delivery_pipeline import (
    PROCESSED_DATA_PATH_CUSTOMER,
    RAW_DATA_DIR,
    run_pipeline,
)

logging.getLogger("matplotlib.category").setLevel(logging.WARNING)

clean_df = run_pipeline(
    output_path=PROCESSED_DATA_PATH_CUSTOMER,
    include_routes=True,
    include_fleet=False,
    include_costs=False,
    include_customers=True,
)

print(f"Loads used for analysis: {len(clean_df)}")

customer_df = pd.read_csv(RAW_DATA_DIR / "customers.csv")
loads_df = pd.read_csv(RAW_DATA_DIR / "loads.csv")
delivery_df = pd.read_csv(RAW_DATA_DIR / "delivery_events.csv")


### 1.1 Verification of the customer merge

In [ ]:
print(f"Loads removed by the pipeline: {len(loads_df) - len(clean_df)}")

display(
    f"Customer data available for {(clean_df['customer_name'].notna().mean()) * 100:.1f}% of loads"
)
display(
    clean_df[
        [
            "customer_name",
            "customer_type",
            "credit_terms_days",
            "primary_freight_type",
            "account_status",
            "contract_start_date",
            "annual_revenue_potential",
        ]
    ].head()
)

*The customer merge is consistent: 100 % of loads have customer data. The 486 loads missing compared wih the raw `loads` table are exactly those removed by the pipeline because of inconsistent timestamps (0.57 %), so the merge itself did not drop any loads.*

## 2. Data Quality Checks
### 2.1 Do all customer_id references resolve?

In [ ]:
display(customer_df["customer_id"].nunique())


def check_reference(relation, child, column, parent, parent_key):
    """Count rows, missing keys and orphans for one foreign key."""
    keys = child[column].dropna()
    return {
        "relation": relation,
        "rows": len(child),
        "missing_keys": child[column].isna().sum(),
        "orphans": (~keys.isin(parent[parent_key])).sum(),
    }


results = pd.DataFrame(
    [
        check_reference(
            "loads.customer_id -> customers",
            clean_df,
            "customer_id",
            customer_df,
            "customer_id",
        ),
    ]
)
results["orphan_share"] = results["orphans"] / results["rows"]
display(results)

#### Interpretation
* All 84,924 `customer_id` references resolve; there are no orphans and no missing keys.
* `customer_id` is unique in the customer table (200 rows, 200 unique IDs) so the join cannot duplicate loads.

### 2.2 Are revenue and charge fields plausible?

In [ ]:
display(
    clean_df[["revenue", "fuel_surcharge", "accessorial_charges"]].describe().round(2)
)
display(clean_df["load_status"].value_counts())

#### Interpretation
* No missing and no negative values in `revenue`, `fuel_surcharge` and `accessorial_charges` (`count` = 84,924 each, `min` >= 0).
* `accessorial_charges` is 0 for at least a quarter of loads (25 % quantile = 0), so there are loads without extra charges, not missing data.
* All 84,924 loads have the status `Completed`, so no filtering by status is needed. 

### 2.3 Are the customer attributes consistent with the loads?

In [ ]:
clean_df["load_date"] = pd.to_datetime(clean_df["load_date"])
before_contract = clean_df["load_date"] < clean_df["contract_start_date"]
inactive = clean_df["account_status"] == "Inactive"

print(f"Loads before contract start: {before_contract.sum()}")
print(f"Share of loads: {before_contract.mean() * 100:.1f} %")
print(f"Loads of inactive customers: {inactive.sum()}")
print(f"Share of loads: {inactive.mean() * 100:.1f} %")


In [ ]:
name_counts = customer_df["customer_name"].value_counts()
print(
    f"Distinct names: {customer_df['customer_name'].nunique()} for {len(customer_df)} customers"
)
print(f"Most frequent name: {name_counts.index[0]} ({name_counts.iloc[0]} customers)")
print(f"Missing values in the customer table: {customer_df.isna().sum().sum()}")

# same name and same categorical attributes (id, potential and start date are left out)
duplicate_names = name_counts[name_counts > 1].index
same_name = customer_df[customer_df["customer_name"].isin(duplicate_names)]
cols_to_check = customer_df.columns.drop(
    ["customer_id", "annual_revenue_potential", "contract_start_date"]
)
display(
    same_name[same_name.duplicated(subset=cols_to_check, keep=False)].sort_values(
        "customer_name"
    )
)

In [ ]:
display(customer_df["annual_revenue_potential"].describe().round(0))
ax = sns.histplot(customer_df["annual_revenue_potential"], bins=20)
ax.set(
    title="Annual Revenue Potential by Customer",
    xlabel="Annual revenue potential ($)",
    ylabel="Number of customers",
)
plt.show()

#### Interpretation

* None of the 84,924 loads took place before the contract start date of its customer, so contract dates and loads are consistent.
* 13,734 loads (16.2 %) belong to customers with `account_status` "Inactive". The data does not show when an account became inactive, so these are most likely real loads of customers that are inactive now.
* `customer_name` is not a key: the 200 customers share only 107 distinct names, so all analyses group by `customer_id`. Three pairs of customers share name and all categorical attributes, but differ in `customer_id`, contract start date and potential; each has its own loads, so they are treated as separate customers.
* `annual_revenue_potential` spreads quite evenly from about 0.1 to 5.0 million \$ (mean 2.69 M, median 2.78 M), so the minimum of 101,641 \$ is not an outlier.

### 2.4 Data Quality Note

* Customer table: 200 customers, `customer_id` is unique, no missing values.
* `customer_name` is not unique (107 names for 200 customers) and is not used as a key.
* Revenue field: no missing or negative values; `accessorial_charges` is 0 for at least a quarter of the loads (no extra charges, not missing). Revenue is about base rate x distance, fuel surcharge is not part of `revenue` (notebook 4).
* Contract start date and account status: no load before the contract start; 16.2 % of the loads belong to inactive customers.
* **Decision:** Keep all 84,924 loads and all 200 customers; group by `customer_id` and use `account_status` as a comparison variable instead of excluding inactive customers; no customers are merged.

## 3. Customer Overview
### 3.1 How is the customer base structured?

#### Customers by Customer Type and Account Status

In [ ]:
display(
    pd.crosstab(
        customer_df["customer_type"], customer_df["account_status"], margins=True
    )
)
display(
    (
        pd.crosstab(
            customer_df["customer_type"],
            customer_df["account_status"],
            normalize="index",
        )
        * 100
    ).round(2)
)

#### Primary Freight Type and Credit Terms

In [ ]:
display(customer_df["primary_freight_type"].value_counts())
display(customer_df["credit_terms_days"].value_counts())

#### Loads per Customer

In [ ]:
customer_loads = clean_df["customer_id"].value_counts()
display(customer_loads.describe().round(1))
print(f"Smallest customer: {customer_loads.idxmin()} with {customer_loads.min()} loads")
print(f"Largest customer: {customer_loads.idxmax()} with {customer_loads.max()} loads")

#### Booking Type and Load Type

In [ ]:
display(
    pd.crosstab(
        clean_df["booking_type"],
        clean_df["load_type"],
        normalize="index",
        margins=True,
    ).round(4)
    * 100
)

display(pd.crosstab(clean_df["customer_type"], clean_df["booking_type"], margins=True))

display(
    pd.crosstab(
        clean_df["customer_type"],
        clean_df["booking_type"],
        normalize="index",
        margins=True,
    ).round(4)
    * 100
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(data=customer_df, x="customer_type", hue="account_status", ax=axes[0])
axes[0].set(
    title="Customers by Type and Account Status",
    xlabel="Customer type",
    ylabel="Number of customers",
)
sns.histplot(clean_df["customer_id"].value_counts(), bins=15, ax=axes[1], kde=True)
axes[1].set(
    title="Loads per Customer", xlabel="Number of loads", ylabel="Number of customers"
)
plt.tight_layout()
plt.show()

#### Interpretation
* The 200 customers split fairly evenly into Contract (75), Spot (63) and Dedicated (62). 168 are active, 32 (16 %) inactive; the inactive share is similar in all three types (13-19 %).
* Loads per customer are very balanced (373-495, mean 425, std 24), so no customer dominates. The share of loads from inactive customers (16.2 %) equals their customer share, so inactive customers ship as many loads as active ones.
* `booking_type` is unrelated to `customer_type`: every customer type books about 25 % Contract, 50 % Dedicated and 25 % Spot loads, and `load_types` is split 50/50 in every booking type. In this simulation, the two are apparently generated independently.
* Automotive is the most frequent freight type (41 customers, 20.5 %), General the least (24, 12 %).
* `credit_terms_days` has four values (15, 30, 45, 60 days) with 41 to 55 customers each, so it can be used as a category in the segment comparison (section 5).

## 4. Revenue by Customer
### 4.1 How much does revenue per load vary between customers?

#### Aggregation per Customer

In [ ]:
customer_stats = (
    clean_df.groupby("customer_id")
    .agg(
        n_loads=("load_id", "count"),
        mean_revenue=("revenue", "mean"),
        total_revenue=("revenue", "sum"),
        total_miles=("actual_distance_miles", "sum"),
    )
    .reset_index()
)

customer_stats["revenue_per_mile"] = (
    customer_stats["total_revenue"] / customer_stats["total_miles"]
)

display(customer_stats.describe().round(2))


#### Distribution of Mean Revenue per Load

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(customer_stats["mean_revenue"], bins=20, kde=True, ax=ax)
ax.axvline(
    customer_stats["mean_revenue"].mean(),
    color="red",
    linestyle="--",
    label=f"Average: {customer_stats['mean_revenue'].mean():.1f}",
)
ax.set(
    title="Mean Revenue per Load by Customer",
    xlabel="Mean revenue per load ($)",
    ylabel="Number of customers",
)
ax.legend()
plt.show()

#### Revenue per Load vs. Revenue per Mile

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=customer_stats, x="n_loads", y="mean_revenue", ax=ax)
ax.set(
    title="Loads per Customer vs. Mean Revenue",
    xlabel="Loads per Customer",
    ylabel="Mean Revenue per Load($)",
)
plt.show()

In [ ]:
spread = (
    customer_stats[["mean_revenue", "revenue_per_mile"]]
    .agg(["min", "max", "mean", "std"])
    .T
)
spread["range"] = spread["max"] - spread["min"]
spread["range_pct_of_mean"] = spread["range"] / spread["mean"] * 100
spread["cv_pct"] = spread["std"] / spread["mean"] * 100
display(spread.round(3))

#### Interpretation
* Mean revenue per load varies between 2,733 and 3,316 $ across customers (range ≈ 19 % of the mean, std 3 %). 
* Per mile the variation is much smaller (≈ 6 % of the mean), so a large part of the spread per load probably comes from different routes and distances, not from different prices. Whether the remaining spread exceeds chance is tested in 4.2

### 4.2 Is that spread larger than expected by chance?

#### Method (permutation test)

*Question:* Is the spread of mean revenue per load between customers larger than expected if the customer had no influence?

1. Compute the observed spread: the standard deviation of the per-customer mean revenue.
2. Shuffle `revenue` randomly across all loads. This breaks any link between customer and revenue, while the number of loads per customer stays the same.
3. Recompute the mean revenue per customer and the standard deviation of these means.
4. Repeat 5,000 times to get the distribution of the spread under "no customer effect".
5. The p-value is the share of shuffled spreads that are at least as large as the observed one (with a +1 correction, so p is never exactly 0).

The same procedure is applied to revenue per mile (per load). Random seed: 42.

#### Permutation Function

In [ ]:
def permutation_spread(values, codes, n_iterations=5000, seed=42):
    """Observed spread (std of group means), null distribution and p-value."""
    counts = np.bincount(codes)
    observed = (np.bincount(codes, weights=values) / counts).std(ddof=1)
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_iterations):
        shuffled = rng.permutation(values)
        null.append((np.bincount(codes, weights=shuffled) / counts).std(ddof=1))
    null = np.array(null)
    p_value = (np.sum(null >= observed) + 1) / (n_iterations + 1)
    return observed, null, p_value

#### Results

In [ ]:
codes, _ = pd.factorize(clean_df["customer_id"])
rev = clean_df["revenue"].to_numpy(dtype=float)
rpm = (clean_df["revenue"] / clean_df["actual_distance_miles"]).to_numpy(dtype=float)

obs_rev, null_rev, p_rev = permutation_spread(rev, codes)
obs_rpm, null_rpm, p_rpm = permutation_spread(rpm, codes)

results = pd.DataFrame(
    {
        "metric": ["Revenue per load", "Revenue per mile (per load)"],
        "observed_std": [obs_rev, obs_rpm],
        "null_mean_std": [null_rev.mean(), null_rpm.mean()],
        "p_value": [p_rev, p_rpm],
    }
).round(4)
display(results)

#### Distribution of Shuffled Spreads

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, title, obs, null in zip(
    axes,
    ["Revenue per load", "Revenue per mile"],
    [obs_rev, obs_rpm],
    [null_rev, null_rpm],
):
    sns.histplot(null, bins=40, ax=ax)
    ax.axvline(obs, color="red", linestyle="--", label=f"Observed: {obs:.4f}")
    ax.set(
        title=title,
        xlabel="Std of customer means (shuffled)",
        ylabel="Number of shuffles",
    )
    ax.legend()
plt.tight_layout()
plt.show()

#### Interpretation
* The spread of mean revenue per load between customers (std 92.3 \$) is almost exactly what a random assignment of loads to customers produces (91.1 \$ on average in the shuffles, p = 0.39). 
* Per mile the result is the same (0.0194 vs. 0.0197, p = 0.59). The borderline impression from the exploratory check does not hold: there is no evidence of a customer effect on revenue.
* The observed spread is only about 1 % above (per load) and 1.5 % below (per mile) the average shuffled spread, so the effect is practically zero.
* A large p-value does not prove that customers are identical, it only means that no difference is detectable. The test looks at the overall spread, not at single customers. Revenue per mile is the mean of the per-load ratios here, so the numbers differ slightly from the ratio of sums in 4.1. The data are simulated, so revenue is probably generated from route and rate independently of the customer.

### 4.3 Does annual_revenue_potential predict actual revenue?

#### Potential vs. Actual Revenue

In [ ]:
customer_potential = customer_stats.merge(
    customer_df[["customer_id", "annual_revenue_potential"]],
    on="customer_id",
    how="left",
    validate="one_to_one",
)

print(
    len(customer_potential), customer_potential["annual_revenue_potential"].isna().sum()
)

# the potential is per year, total_revenue covers the whole data period
years = (clean_df["load_date"].max() - clean_df["load_date"].min()).days / 365.25
customer_potential["revenue_per_year"] = customer_potential["total_revenue"] / years
customer_potential["actual_vs_potential_pct"] = (
    customer_potential["revenue_per_year"]
    / customer_potential["annual_revenue_potential"]
    * 100
)

print(f"Data period: {years:.1f} years")
display(
    customer_potential[
        ["annual_revenue_potential", "revenue_per_year", "actual_vs_potential_pct"]
    ]
    .describe()
    .round(1)
)

In [ ]:
rows = []
for col in ["total_revenue", "mean_revenue", "n_loads"]:
    rho, p = stats.spearmanr(
        customer_potential["annual_revenue_potential"], customer_potential[col]
    )
    rows.append({"metric": col, "spearman_rho": rho, "p_value": p})
display(pd.DataFrame(rows).round(3))

In [ ]:
rho_total, _ = stats.spearmanr(
    customer_potential["annual_revenue_potential"], customer_potential["total_revenue"]
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=customer_potential, x="annual_revenue_potential", y="total_revenue", ax=ax
)
ax.set(
    title=f"Revenue Potential vs. Actual Revenue (Spearman rho = {rho_total:.2f})",
    xlabel="Annual revenue potential ($)",
    ylabel="Total revenue in data period ($)",
)
plt.show()

#### Interpretation
* The Spearman correlation between `annual_revenue_potential` and actual revenue is close to zero and not significant: total revenue rho = -0.08 (p = 0.27), mean revenue per load rho = −0.11 (p = 0.13) and number of loads rho = −0.04 (p = 0.61). So the potential does not predict actual revenue.
* Actual revenue per year is almost the same for all customers (mean 438,000 \$, std 30,000 \$, range 359,000–515,000 \$), while the potential spreads from 0.1 to 5.0 million \$. For the typical customer the actual revenue is only about 16 % of the potential (median; the mean of 33 % is pulled up by a few customers with a small potential), and for customers with a low potential it is a multiple of it (up to 480 %).
* The origin of the potential is unknown; in this simulated data it seems to be generated independently of the loads. With 200 customers only fairly strong relations (|rho| > 0.14) can be detected, and revenue excludes fuel surcharge and accessorial charges.

## 5. Revenue by Segment

### 5.1 Does revenue differ by booking type, load type, customer type or credit terms?

#### Summary by Segment

In [ ]:
clean_df["total_billed"] = clean_df["revenue"] + clean_df["fuel_surcharge"]
segment_cols = ["booking_type", "load_type", "customer_type", "credit_terms_days"]


def segment_summary(df, col):
    out = (
        df.groupby(col)
        .agg(
            n_loads=("load_id", "count"),
            mean_revenue=("revenue", "mean"),
            total_revenue=("revenue", "sum"),
            total_miles=("actual_distance_miles", "sum"),
            mean_distance=("actual_distance_miles", "mean"),
            mean_fuel_surcharge=("fuel_surcharge", "mean"),
            mean_acc_charges=("accessorial_charges", "mean"),
            mean_billed=("total_billed", "mean"),
        )
        .reset_index()
    )
    out["revenue_per_mile"] = out["total_revenue"] / out["total_miles"]
    return out


segment_tables = {col: segment_summary(clean_df, col) for col in segment_cols}
for col, table in segment_tables.items():
    display(table.round(2))


#### How Does the Fuel Surcharge Relate to Revenue?

In [ ]:
ratio = clean_df["fuel_surcharge"] / (
    clean_df["fuel_surcharge_rate"] * clean_df["typical_distance_miles"]
)
print(
    f"surcharge / (rate x typical distance): min {ratio.min():.3f}, max {ratio.max():.3f}"
)

display(
    clean_df[["revenue", "fuel_surcharge", "actual_distance_miles"]].corr().round(3)
)

#### Charts

In [ ]:
overall_rpm = clean_df["revenue"].sum() / clean_df["actual_distance_miles"].sum()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for ax, col in zip(axes.ravel(), segment_cols):
    sns.barplot(
        data=segment_tables[col], x=col, y="revenue_per_mile", ax=ax, color="C0"
    )
    ax.axhline(
        overall_rpm, color="red", linestyle="--", label=f"Overall: {overall_rpm:.2f}"
    )
    ax.set(
        title=f"Revenue per Mile by {col}", xlabel=col, ylabel="Revenue per mile ($)"
    )
    ax.legend()
plt.tight_layout()
plt.show()

for col in segment_cols:
    rpm = segment_tables[col]["revenue_per_mile"]
    print(
        f"{col}: {rpm.min():.3f} to {rpm.max():.3f} (difference {(rpm.max() - rpm.min()) / overall_rpm:.1%} of overall)"
    )

In [ ]:
rpm_load = (clean_df["revenue"] / clean_df["actual_distance_miles"]).to_numpy(
    dtype=float
)
for col in segment_cols:
    codes, _ = pd.factorize(clean_df[col])
    obs, null, p = permutation_spread(rpm_load, codes)
    print(f"{col}: observed {obs:.4f}, shuffled {null.mean():.4f}, p = {p:.3f}")

#### Interpretation

* Revenue per mile is practically identical in all segments: 2.15 \$ for every group, with a maximum difference of 0.1–0.4 % between the groups of a variable. The permutation tests confirm that these differences are consistent with chance (p = 0.12 to 0.82, all above 0.05).
* The fuel surcharge equals rate × typical distance for every load (ratio = 1.000) and is not part of `revenue`. The strong correlation between revenue and surcharge (r = 0.85) comes from distance (r = 0.93 between revenue and distance, r = 0.88 between surcharge and distance), so the segments are compared per mile.
* Mean distance (1,431–1,444 miles) and mean revenue per load (3,077–3,101 \$) are also almost identical across the segments, so the route mix does not differ between them either. Since the data are simulated, a missing difference mainly shows that the simulation did not build one in.


### 5.2 Does the primary freight type match the load type?

In [ ]:
display(
    (
        pd.crosstab(
            clean_df["primary_freight_type"], clean_df["load_type"], normalize="index"
        )
        * 100
    ).round(1)
)

#### Interpretation
* `load_type` is split about 50/50 (49.2–50.8 %) within every freight type, which is exactly the chance level for two categories. Even Food/Beverage customers do not ship more refrigerated loads (49.8 % Refrigerated), so there is no relation between `primary_freight_type` and `load_type`.
* The deviations of up to 0.8 percentage points are within normal random variation for 10,000–17,000 loads per freight type. In this simulation, the load type is apparently assigned independently of the customer's freight type.

## 6. Service Level by Customer

### 6.0 Event-level Table (for 6.1 and 6.2)
*Built in the notebook because only this notebook needs one row per event with the customer attached; the pipeline stays at one row per load.*

In [ ]:
events = delivery_df[delivery_df["load_id"].isin(clean_df["load_id"])].copy()

events = events.merge(
    clean_df[["load_id", "customer_id"]],
    on="load_id",
    how="left",
    validate="many_to_one",
)

print(len(events), 2 * len(clean_df), len(events) == 2 * len(clean_df))
print("Events without customer:", events["customer_id"].isna().sum())


### 6.1 Does on-time rate / detention differ by customer?

#### Aggregation per Customer

In [ ]:
customer_service = (
    events.groupby("customer_id")
    .agg(
        n_events=("event_id", "count"),
        on_time_events=("on_time_flag", "sum"),
        total_detention=("detention_minutes", "sum"),
    )
    .reset_index()
    .assign(
        on_time_rate_pct=lambda df: df["on_time_events"] / df["n_events"] * 100,
        avg_det_mins=lambda df: df["total_detention"] / df["n_events"],
    )
    .assign(
        se_on_time_pct=lambda df: np.sqrt(
            df["on_time_rate_pct"] * (100 - df["on_time_rate_pct"]) / df["n_events"]
        )
    )
)

display(customer_service.describe().round(2))

In [ ]:
pickup_share = events["event_type"].eq("Pickup").groupby(events["customer_id"]).mean()
display(pickup_share.describe())

#### On-Time Rate and Detention per Customer

In [ ]:
# 1. Margin of error (95 % confidence interval)
customer_service = customer_service.assign(
    margin_of_error_pct=lambda df: 1.96 * df["se_on_time_pct"]
)

# 2. Overall rate over all events
overall_rate_pct = events["on_time_flag"].mean() * 100

outside = (
    customer_service["on_time_rate_pct"] - overall_rate_pct
).abs() > customer_service["margin_of_error_pct"]
print(
    f"{outside.sum()} of {len(customer_service)} customers have an interval that does not "
    f"contain the overall rate (expected by chance: about {len(customer_service) * 0.05:.0f})"
)

# 3. Select top 15 and bottom 15 customers
top_15 = customer_service.nlargest(15, "on_time_rate_pct")
bottom_15 = customer_service.nsmallest(15, "on_time_rate_pct")


plot_df = (
    pd.concat([bottom_15, top_15])
    .drop_duplicates(subset=["customer_id"])
    .sort_values("on_time_rate_pct", ascending=True)
)

# 4. Plot
plt.figure(figsize=(10, 10))


plt.errorbar(
    x=plot_df["on_time_rate_pct"],
    y=plot_df["customer_id"].astype(str),
    xerr=plot_df["margin_of_error_pct"],
    fmt="o",
    color="skyblue",
    ecolor="gray",
    capsize=4,
    linewidth=1.5,
    label="Customer Rate 95% CI",
)


plt.axvline(
    x=overall_rate_pct,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Overall Rate ({overall_rate_pct:.1f}%)",
)


plt.title(
    "Top 15 and Bottom 15 Customers: On-Time Rate with 95 % Confidence Interval",
)
plt.xlabel("On-Time Rate (%)")
plt.ylabel("Customer ID")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower right")
plt.tight_layout()

plt.show()

In [ ]:
# overall_on_time_rate = customer_service["on_time_rate_pct"].mean()
overall_avg_det = events["detention_minutes"].mean()
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=customer_service, x="n_events", y="on_time_rate_pct", ax=ax)
plt.axhline(
    y=overall_rate_pct,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Overall Rate ({overall_rate_pct:.2f} %)",
)

ax.set_xlabel("Number of Events")
ax.set_ylabel("On-Time Rate (%)")
ax.set_title("Number of Events vs. On-Time-Rate per Customer")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=customer_service, x="n_events", y="avg_det_mins", ax=ax)

plt.axhline(
    y=overall_avg_det,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Overall Avg. Detention ({overall_avg_det:.2f} min)",
)

ax.set_xlabel("Number of Events")
ax.set_ylabel("Avg. Detention Time (mins)")
ax.set_title("Number of Events vs. Avg. Detention Time (mins)")
ax.legend()
plt.show()

#### By Customer Type and Account Status

In [ ]:
customer_service_expanded = customer_service.merge(
    customer_df[["customer_id", "account_status", "customer_type"]],
    on="customer_id",
    how="left",
    validate="one_to_one",
)


def service_summary(df, col):
    out = (
        df.groupby(col)
        .agg(
            n_customers=("customer_id", "count"),
            n_events=("n_events", "sum"),
            on_time_events=("on_time_events", "sum"),
            total_detention=("total_detention", "sum"),
        )
        .reset_index()
    )
    out["on_time_rate_pct"] = out["on_time_events"] / out["n_events"] * 100
    out["avg_det_mins"] = out["total_detention"] / out["n_events"]
    return out


service_cols = ["customer_type", "account_status"]
service_tables = {
    col: service_summary(customer_service_expanded, col) for col in service_cols
}
for table in service_tables.values():
    display(table.round(2))

overall_det = events["detention_minutes"].mean()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for j, col in enumerate(service_cols):
    t = service_tables[col]
    sns.barplot(data=t, x=col, y="on_time_rate_pct", ax=axes[0, j], color="C0")
    axes[0, j].axhline(
        overall_rate_pct,
        color="red",
        linestyle="--",
        label=f"Overall: {overall_rate_pct:.1f} %",
    )
    axes[0, j].set(
        title=f"On-Time Rate by {col}", xlabel=col, ylabel="On-time rate (%)"
    )
    axes[0, j].legend()
    sns.barplot(data=t, x=col, y="avg_det_mins", ax=axes[1, j], color="C0")
    axes[1, j].axhline(
        overall_det,
        color="red",
        linestyle="--",
        label=f"Overall: {overall_det:.1f} min",
    )
    axes[1, j].set(
        title=f"Avg. Detention by {col}", xlabel=col, ylabel="Avg. detention (min)"
    )
    axes[1, j].legend()
plt.tight_layout()
plt.show()


#### Interpretation
* The on-time rate per customer ranges from 50.0 % to 61.0 % (mean 55.7 %, std 1.6 pp), the average detention from 85.9 to 98.8 min (mean 91.5, std 2.3 min). With about 850 events per customer, the standard error of a single rate is also about 1.7 pp, so the spread between customers is as large as random fluctuation alone; 8 of the 200 customers have a confidence interval that does not contain the overall rate, which is close to the 10 expected by chance (5 %).
* The groups differ by less than 0.6 pp in on-time rate and 1.2 min in detention (customer type) and by 0.2 pp / 0.2 min (account status).
* `on_time_flag` means "within 2 hours of the schedule", not "not late". Pickups (66.7 %) and deliveries (44.7 %) differ strongly, but every customer has exactly 50 % pickup events, so the event mix does not explain differences between customers.

### 6.2 Is that spread larger than expected by chance?

*Same permutation test as in 4.2, applied to `on_time_flag` and `detention_minutes` with customers as groups (5,000 iterations, seed 42).*

In [ ]:
codes, _ = pd.factorize(events["customer_id"])
ote = events["on_time_flag"].to_numpy(dtype=float)
det = events["detention_minutes"].to_numpy(dtype=float)

obs_ote, null_ote, p_ote = permutation_spread(ote, codes)
obs_det, null_det, p_det = permutation_spread(det, codes)

results = pd.DataFrame(
    {
        "metric": ["On-Time Rate", "Detention Minutes"],
        "observed_std": [obs_ote, obs_det],
        "null_mean_std": [null_ote.mean(), null_det.mean()],
        "p_value": [p_ote, p_det],
    }
).round(4)
display(results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, title, obs, null in zip(
    axes,
    ["On-Time Rate", "Detention Minutes"],
    [obs_ote, obs_det],
    [null_ote, null_det],
):
    sns.histplot(null, bins=40, ax=ax)
    ax.axvline(obs, color="red", linestyle="--", label=f"Observed: {obs:.4f}")
    ax.set(
        title=title,
        xlabel="Std of customer means (shuffled)",
        ylabel="Number of shuffles",
    )
    ax.legend()
plt.tight_layout()
plt.show()

#### Interpretation
* The spread of on-time rates between customers (std 1.64 pp) is slightly below what a random assignment of events produces (1.71 pp, p = 0.77), and the same holds for detention (2.33 vs. 2.36 min, p = 0.58). Customers therefore do not differ in service level beyond chance, which matches the result for revenue in 4.2.
* The test looks at the overall spread, not at single customers. The two events of a load are not independent, which makes the effective sample smaller, but with such large p-values this does not change the conclusion. The data are simulated, so a missing difference mainly shows that the simulation did not build one in.

## 7. Revenue Concentration

### 7.1 How much of total revenue comes from the top customers?

#### Ranking and Cumulative Share

In [ ]:
customer_concentration = (
    customer_stats[["customer_id", "total_revenue"]]
    .sort_values("total_revenue", ascending=False)
    .reset_index(drop=True)
    .assign(
        revenue_rank=lambda df: df.index + 1,
        cum_customers_pct=lambda df: df["revenue_rank"] / len(df) * 100,
        cum_revenue_pct=lambda df: (
            df["total_revenue"].cumsum() / df["total_revenue"].sum() * 100
        ),
    )
)

for share in [10, 20]:
    n_top = int(len(customer_concentration) * share / 100)
    revenue_pct = customer_concentration.loc[n_top - 1, "cum_revenue_pct"]
    print(
        f"Top {share} % of customers ({n_top}) generate {revenue_pct:.1f} % of revenue"
    )

#### Cumulative Revenue Curve

In [ ]:
lorenz = customer_concentration.sort_values("total_revenue").reset_index(drop=True)
x_l = np.insert(((lorenz.index + 1) / len(lorenz)).to_numpy(), 0, 0)
y_l = np.insert(
    (lorenz["total_revenue"].cumsum() / lorenz["total_revenue"].sum()).to_numpy(), 0, 0
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(x_l * 100, y_l * 100, color="blue", linewidth=2.5, label="Lorenz curve")
ax.plot([0, 100], [0, 100], color="red", linestyle="--", label="Equal distribution")
ax.fill_between(x_l * 100, y_l * 100, x_l * 100, color="skyblue", alpha=0.3)
ax.set(
    title="Lorenz Curve of Revenue by Customer",
    xlabel="Share of customers, smallest first (%)",
    ylabel="Cumulative share of revenue (%)",
    xlim=(0, 100),
    ylim=(0, 100),
)
ax.legend(loc="upper left")
plt.show()

area = np.sum((y_l[1:] + y_l[:-1]) / 2 * np.diff(x_l))
print(f"Gini coefficient: {1 - 2 * area:.3f}")

In [ ]:
largest_share = (
    customer_concentration.loc[0, "total_revenue"]
    / customer_concentration["total_revenue"].sum()
    * 100
)
print(f"Largest customer: {largest_share:.2f} % of revenue")

#### Compared with a Random Assignment (optional)

In [ ]:
codes, _ = pd.factorize(clean_df["customer_id"])
rev = clean_df["revenue"].to_numpy(dtype=float)
n_customers = codes.max() + 1
n_top = int(n_customers * 0.10)
rng = np.random.default_rng(42)


def top_share(codes_):
    totals = np.bincount(codes_, weights=rev, minlength=n_customers)
    return np.sort(totals)[::-1][:n_top].sum() / totals.sum() * 100


observed = top_share(codes)
null = np.array([top_share(rng.permutation(codes)) for _ in range(5000)])
p = ((null >= observed).sum() + 1) / (len(null) + 1)
print(f"Observed: {observed:.2f} %, shuffled: {null.mean():.2f} %, p = {p:.3f}")

#### Revenue from Inactive Accounts

In [ ]:
inactive_ids = customer_df.loc[
    customer_df["account_status"] == "Inactive", "customer_id"
]
inactive_revenue = customer_stats.loc[
    customer_stats["customer_id"].isin(inactive_ids), "total_revenue"
].sum()
print(
    f"Inactive customers: {inactive_revenue / customer_stats['total_revenue'].sum() * 100:.1f} % of revenue"
)

#### Interpretation
* The top 10 % of customers (20) generate 11.2 % of the total revenue, the top 20 % (40) 21.9 %; the curve lies close to the diagonal, so revenue is spread almost evenly and there is no relevant customer concentration risk (the largest customer has 0.59 %).
* The reason is that all customers have a similar number of loads (373–495) and the same revenue per load (4.1, 4.2), so no customer can dominate in this simulation.
* Customers with an inactive account generate 16.2 % if the revenue (16 % of the customers), so they contibute as much as their number suggests.
* If the loads are assigned to customers at random, the top 10 % generate on average 11.17 % of the revenue (observed: 11.2 %, p = 0.309), so the concentration is within what chances produces.


## 8. Conclusion

*Customer base (3): The 200 customers are split fairly evenly into Contract (75), Spot (63) and Dedicated (62); 16 % have an inactive account. All customers have a similar number of loads (373–495), and the booking type is unrelated to the customer type.*

*Revenue between customers (4.1, 4.2): Mean revenue per load varies by only about 3 % between customers (2,733–3,316 $), per mile by about 1 %. A permutation test shows that this spread is what a random assignment of loads would produce (p = 0.39 per load, p = 0.59 per mile), so there is no customer effect on revenue.*

*Revenue potential vs. actual revenue (4.3): `annual_revenue_potential` does not predict actual revenue (Spearman rho between −0.11 and −0.04, all p > 0.1). Every customer bills about 438,000 $ per year, so the median customer reaches only about 16 % of its potential.*

*Segments (5): Revenue per mile differs by at most 0.4 % between the groups of booking type, load type, customer type and credit terms (permutation tests p = 0.12–0.82). The strong correlation between revenue and fuel surcharge (r = 0.85) is a distance effect, because the surcharge is rate × typical distance and not part of revenue. The primary freight type of a customer has no relation to the load type (about 50/50 in every group).*

*Service level by customer (6.1, 6.2): On-time rates (50.0–60.7 %) and detention (85.9–98.8 min) vary between customers as much as random fluctuation with about 850 events per customer would (std 1.6 pp vs. standard error 1.7 pp; permutation tests p = 0.77 and p = 0.58). Customer type and account status make no relevant difference (< 0.6 pp, < 1.2 min).*

*Revenue concentration (7): Revenue is spread almost evenly: the top 10 % of customers generate 11.2 % of the revenue, the top 20 % generate 21.9 %, and the largest customer 0.59 %. There is no customer concentration risk.*

*Limitations: The data are simulated, so the missing differences mainly show that the simulation did not build customer effects in. The permutation tests look at the overall spread, not at single customers, and the two events of a load are not independent. Revenue excludes fuel surcharge and accessorial charges. Inactive accounts are kept in all analyses, because the data does not show when an account became inactive.*

*Next: [06_driver_safety_analysis](06_driver_safety_analysis.ipynb) (optional), then [07_correlations](07_correlations.ipynb).*

## Appendix: Export for Dashboard 05

Aggregated tables for the Power BI dashboard (`dashboard/05_customer_analysis/data/`). The weighted measures (on-time rate, detention, revenue per mile) are computed in Power BI from the sums.

In [ ]:
from delivery_pipeline import PROJECT_ROOT

export_dir = PROJECT_ROOT / "dashboard" / "05_customer_analysis" / "data"
export_dir.mkdir(parents=True, exist_ok=True)

customer_export = customer_stats.merge(
    customer_df[
        [
            "customer_id",
            "customer_type",
            "account_status",
            "primary_freight_type",
            "credit_terms_days",
            "annual_revenue_potential",
        ]
    ],
    on="customer_id",
    how="left",
    validate="one_to_one",
).merge(
    clean_df.groupby("customer_id", as_index=False).agg(
        total_billed=("total_billed", "sum")
    ),
    on="customer_id",
    how="left",
    validate="one_to_one",
)

segment_stats = pd.concat(
    [
        table.rename(columns={col: "value"}).assign(segment=col).astype({"value": str})
        for col, table in segment_tables.items()
    ],
    ignore_index=True,
)

exports = {
    "customer_stats": customer_export,
    "customer_service": customer_service[
        ["customer_id", "n_events", "on_time_events", "total_detention"]
    ],
    "segment_stats": segment_stats,
    "customer_concentration": customer_concentration,
}
for name, table in exports.items():
    table.to_csv(export_dir / f"{name}.csv", index=False)
    print(f"{name}: {len(table)} rows")
